# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [ ]:
import glob
import os
import wer
import observation_model
import openfst_python as fst
from decoder import MyViterbiDecoder
from utils import generate_word_wfst, parse_lexicon, generate_symbol_tables, generate_phone_wfst
import math
# ... (add your code to create WFSTs and Viterbi Decoder)

def create_wfst(lexicon, word_table, phone_table, state_table, n_states=3):
    """
    Simplest WFST: any sequence of words from the lexicon.
    Input labels: HMM state IDs (e.g. 'p_1'), matching ObservationModel.
    Output labels: words, on the last transition of each word.
    """
    f = fst.Fst('log')

    # Attach symbol tables so the decoder can map IDs <-> strings
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)

    loop_state = f.add_state()
    f.set_start(loop_state)
    f.set_final(loop_state)  # allow any number of words, including zero

    for word, phones in lexicon.items():
        word_id = word_table.find(word)

        current_state = loop_state

        for phone_idx, phone in enumerate(phones):
            for i in range(1, n_states + 1):
                state_sym = f"{phone}_{i}"
                in_label = state_table.find(state_sym)

                # Self-loop
                sl_weight = fst.Weight('log', -math.log(0.5))
                f.add_arc(current_state, fst.Arc(in_label, 0, sl_weight, current_state))

                # Forward transition
                next_state = f.add_state()
                fw_weight = fst.Weight('log', -math.log(0.5))

                # Put the word label only on the very last transition of the word
                if phone_idx == len(phones) - 1 and i == n_states:
                    out_label = word_id
                else:
                    out_label = 0

                f.add_arc(current_state, fst.Arc(in_label, out_label, fw_weight, next_state))
                current_state = next_state

        # After finishing this word, return to loop_state (epsilon transition)
        f.add_arc(current_state, fst.Arc(0, 0, fst.Weight.One(f.weight_type()), loop_state))

    return f


def read_transcription(wav_file):
    """
    Get the transcription corresponding to wav_file.
    """
    transcription_file = os.path.splitext(wav_file)[0] + '.txt'
    
    with open(transcription_file, 'r') as f:
        transcription = f.readline().strip()
    
    return transcription

lex = parse_lexicon("lexicon.txt")
word_table, phone_table, state_table = generate_symbol_tables(lex)

f = create_wfst(lex, word_table, phone_table, state_table)
print(f)
#lex = parse_lexicon("lexicon.txt")
#word_table, phone_table, state_table = generate_symbol_tables(lex)

total_substitutions = 0
total_deletions = 0
total_insertions = 0
total_words = 0

for wav_file in glob.glob('/group/teaching/asr/labs/recordings/*.wav'):    # replace path if using your own
                                                                           # audio files
    
    decoder = MyViterbiDecoder(f, wav_file)
    
    decoder.decode()
    (state_path, words) = decoder.backtrace()  # you'll need to modify the backtrace() from Lab 4
                                               # to return the words along the best path
    print(words)
    transcription = read_transcription(wav_file)
    error_counts = wer.compute_alignment_errors(transcription, words)
    word_count = len(transcription.split())
    
    print(error_counts, word_count)     # you'll need to accumulate these to produce an overall Word Error Rate

    total_substitutions += error_counts[0]
    total_deletions += error_counts[1]
    total_insertions += error_counts[2]
    total_words += word_count
    
    print(f"File: {os.path.basename(wav_file)} | Errors (S,D,I): {error_counts} | Words: {word_count}")
    
overall_wer = (total_substitutions + total_deletions + total_insertions) / total_words
print(f"\nOverall Word Error Rate (WER): {overall_wer:.2%}")

0	0	ey_1	<eps>
0	1	ey_1	<eps>	0.105361
0	0	ah_1	<eps>
0	4	ah_1	<eps>	0.105361
0	0	p_1	<eps>
0	10	p_1	<eps>	0.105361
0	0	p_1	<eps>
0	19	p_1	<eps>	0.105361
0	0	p_1	<eps>
0	34	p_1	<eps>	0.105361
0	0	p_1	<eps>
0	46	p_1	<eps>	0.105361
0	0	p_1	<eps>
0	58	p_1	<eps>	0.105361
0	0	p_1	<eps>
0	76	p_1	<eps>	0.105361
0	0	dh_1	<eps>
0	88	dh_1	<eps>	0.105361
0	0	w_1	<eps>
0	94	w_1	<eps>	0.105361
0
1	1	ey_2	<eps>
1	2	ey_2	<eps>	0.105361
2	2	ey_3	<eps>
2	3	ey_3	a	0.105361
3	0	<eps>	<eps>
4	4	ah_2	<eps>
4	5	ah_2	<eps>	0.105361
5	5	ah_3	<eps>
5	6	ah_3	<eps>	0.105361
6	6	v_1	<eps>
6	7	v_1	<eps>	0.105361
7	7	v_2	<eps>
7	8	v_2	<eps>	0.105361
8	8	v_3	<eps>
8	9	v_3	of	0.105361
9	0	<eps>	<eps>
10	10	p_2	<eps>
10	11	p_2	<eps>	0.105361
11	11	p_3	<eps>
11	12	p_3	<eps>	0.105361
12	12	eh_1	<eps>
12	13	eh_1	<eps>	0.105361
13	13	eh_2	<eps>
13	14	eh_2	<eps>	0.105361
14	14	eh_3	<eps>
14	15	eh_3	<eps>	0.105361
15	15	k_1	<eps>
15	16	k_1	<eps>	0.105361
16	16	k_2	<eps>
16	17	k_2	<eps>	0.105361
17	17	k_3	<eps>
17	18	k_3	pec

Exception: 